In [1]:
import sys
sys.path.append("../..")

In [2]:
from syntax_tokenizer import SyntaxTokenizer
from model import ModelConfig, LlamaModel
from train import TrainerConfig, SimpleDataLoader, Trainer

In [3]:
with open("../../data/complete_shakespeare.txt") as f:
    text = f.read()

In [4]:
texts = text.split("\n\n\n\n")

In [5]:
# tokenizer = SyntaxTokenizer()
# tokenizer.train(texts)
tokenizer = SyntaxTokenizer.load("./shakespeare_bidirectional/shakespeare_tokenizer.data")

In [6]:
tokenizer.save("./shakespeare_bidirectional/shakespeare_tokenizer.data")

In [7]:
tokenizer.vocab_size

12853

In [8]:
model_config = ModelConfig(
    is_causal=False,
    vocab_size=tokenizer.vocab_size,
    d_model=576,
    d_head=64,
    d_mlp_proj=1536,
    n_layers=30,
    n_kv_heads=3,
    n_attn_heads=9,
    rms_norm_eps=1e-5,
    initializer_range=0.02,
    rope_theta=100000.0,
    padding_idx=tokenizer.data.pad_token_id
)

In [9]:
train_config = TrainerConfig(
    is_causal=False,
    mask_ratio=0.15,
    per_device_train_batch_size=32,
    max_seq_len=512,
    num_epochs=32,
    eval_interval_steps=25,
    learning_rate=4e-3,
    grad_clip_norm=1.0,
    val_size=0.05,
    log_dir="runs/shakespeare_bidirectional",
    warmup_ratio=0.1
)

In [10]:
n = 100
size_patch = len(text) // 100
texts_equal = [text[i:i+size_patch] for i in range(n+1)]
dataloader = SimpleDataLoader(train_config, tokenizer, texts=texts_equal)

Total tokens                   | 1,341,886


In [11]:
model = LlamaModel(model_config)
trainer = Trainer(train_config, model, tokenizer)

Num Trainable Params           | 121,010,112
Train device                   | cuda, NVIDIA GeForce RTX 3090, N=1
Training precision             | torch.bfloat16
Flash Attention                | True
torch.compile()                | True
DistributedDataParallel        | False
Batch size                     | 2,457




In [12]:
trainer.train(dataloader)

Training steps                 | 2,496 
Step: 0, Training Loss: 9.62407, LR: 0.0002000, Tokens/sec: 706.90
Step: 1, Training Loss: 8.74499, LR: 0.0002153, Tokens/sec: 758.10
Step: 2, Training Loss: 8.07327, LR: 0.0002305, Tokens/sec: 116418.35
Step: 3, Training Loss: 7.72003, LR: 0.0002458, Tokens/sec: 111916.83
Computing Eval loss, steps: 5
Step: 3, Eval Loss: 7.48310
Step: 4, Training Loss: 7.53650, LR: 0.0002610, Tokens/sec: 104303.69
Step: 5, Training Loss: 7.42135, LR: 0.0002763, Tokens/sec: 111659.29
Step: 6, Training Loss: 7.10119, LR: 0.0002916, Tokens/sec: 111592.30
Step: 7, Training Loss: 6.95208, LR: 0.0003068, Tokens/sec: 108697.15
Step: 8, Training Loss: 6.70528, LR: 0.0003221, Tokens/sec: 108239.02
Step: 9, Training Loss: 6.49253, LR: 0.0003373, Tokens/sec: 106597.43
Step: 10, Training Loss: 6.27784, LR: 0.0003526, Tokens/sec: 106445.98
Step: 11, Training Loss: 5.90536, LR: 0.0003679, Tokens/sec: 106663.49
Step: 12, Training Loss: 5.77862, LR: 0.0003831, Tokens/sec: 10703

In [13]:
trainer.save_checkpoint("shakespeare_bidirectional")

Saving checkpoint: shakespeare_bidirectional/model.checkpoint.2025-04-22--01-37-06.pt
Checkpoint saved
